# Jak Wojtek uczy się chodzić

Wojtek to czworonożny robot, który zaczął jako **4BarBot** na Politechnice Wrocławskiej. Chodu nie ma zaprogramowanego: uczy się go metodą uczenia ze wzmocnieniem (RL) w symulacji, a wytrenowana sieć trafia potem na fizycznego robota. Ten notebook prowadzi przez tę drogę krok po kroku: robot w symulatorze MuJoCo, środowisko treningowe, trening, eksport polityki i na końcu porównanie Twojej polityki z tą, która jeździ na robocie. Każdy krok kończy się widokiem z MuJoCo.

## Robot

- 4 nogi, w każdej 3 silniki: odwodzenie biodra, biodro, kolano. Razem 12 silników sterowanych **pozycyjnie**: polityka podaje kąt docelowy, regulator PD w napędzie (na robocie MD80, w MuJoCo ten sam model serwa) zamienia go na moment.
- Nogi są czworobokami przegubowymi: poniżej silników łańcuch kinematyczny się zamyka. Za osobliwością kolana (ok. 3,2 rad) mechanizm może się przeskoczyć, dlatego cel kolana jest zawsze ograniczony.
- Masa 14 kg, wysokość stania ok. 0,125 m.
- Fizyka liczy się 250 razy na sekundę (krok 4 ms), polityka działa 50 razy na sekundę: jedna decyzja = 5 kroków fizyki.

## Skąd jest model

Wszystko leży w pakiecie ROS `ros/src/wojtek_description/`:

- `meshes/*.stl` — geometria ogniw robota z CAD-u;
- `urdf/*.urdf.xacro` — opis robota dla ROS-a (ten, którego używa prawdziwy robot i RViz);
- `mujoco/wojtek.xml` — ten sam robot zapisany w formacie MuJoCo (MJCF): ogniwa, przeguby, domknięcia czworoboków, siatki z `meshes/`, silniki i czujniki. Jest źródłem dla treningu;
- `mujoco/wojtek_mjx.xml` + `scene_mjx.xml` — wersja treningowa, **generowana** poleceniem `./training/run.sh build`. Skrypt bierze `wojtek.xml` i nanosi zmiany potrzebne w treningu: siatki przestają kolidować (stopy dostają kule, korpus prostopadłościan), korpus dostaje jawną masę, 12 silników momentowych staje się serwami PD, krok fizyki ustawiony na 4 ms. Tych plików nie edytuje się ręcznie; `scene_mjx.xml` dokłada podłogę, światło i kamerę śledzącą.

Notebook ładuje właśnie `scene_mjx.xml`, czyli dokładnie to, na czym trenuje polityka.

Środowisko Colab: **GPU** (Runtime → Change runtime type). Uruchamiaj komórki po kolei.

## Krok 0 — Instalacja

Klonuje repozytorium i instaluje `training/` (JAX, MuJoCo MJX, MJWarp, Brax). Jeśli następna komórka nie zaimportuje bibliotek, zrób Runtime → Restart session i uruchom od początku.

In [ ]:
import os, subprocess, sys
from pathlib import Path

if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")   # renderowanie bez ekranu; przed `import mujoco`

REPO_BRANCH = "Add-notebook-with-the-guidance-how-to-train-Wojtek"   # po scaleniu: "main"
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "training" / "run.sh").exists()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd() / "w01-tek"
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "-q", "-b", REPO_BRANCH, "https://github.com/machinekind/w01-tek.git", str(REPO_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "-q", "--ff-only"])   # ponowne uruchomienie: dociągnij zmiany
TRAINING = REPO_ROOT / "training"

try:
    import wojtek_rl, fast_simplification, trimesh  # noqa: F401
except ImportError:
    import tomllib
    lock = tomllib.loads((TRAINING / "uv.lock").read_text())
    mujoco_pin = next(p["version"] for p in lock["package"] if p["name"] == "mujoco")   # ta sama wersja co w locku
    res = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(TRAINING), f"mujoco=={mujoco_pin}",
                          "trimesh", "fast-simplification"],   # dwa ostatnie: uproszczone siatki do renderowania
                         capture_output=True, text=True)
    if res.returncode:
        print(res.stdout[-1500:], res.stderr[-3000:])
        raise SystemExit("pip install nie powiodł się (patrz wyżej)")
    # Colab ma preinstalowany nowszy plugin JAX dla CUDA 13; obok jax 0.9.2 tylko generuje błędy.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax-cuda13-plugin", "jax-cuda13-pjrt"], capture_output=True)
    print("zainstalowano; jeśli następna komórka nie działa, zrestartuj sesję i uruchom od początku")
print(REPO_ROOT, "| python", sys.version.split()[0])

In [ ]:
import shutil
import mediapy as media
import mujoco
import numpy as np

if shutil.which("ffmpeg") is None:          # mediapy potrzebuje binarki ffmpeg
    import imageio_ffmpeg
    media.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

for p in (TRAINING, REPO_ROOT / "learning"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from wojtek_rl import paths
from lowpoly import render_model

# Colab nie ma OpenGL od NVIDII: MuJoCo renderuje programowo, a pełne siatki robota
# (600 tys. trójkątów) kosztują 0,8 s na klatkę. Rysujemy więc kopię modelu z uproszczonymi
# siatkami; fizyka liczy się na oryginale.
CACHE = TRAINING / "videos" / "guide" / "lowpoly"
rmodel = render_model(paths.SCENE_XML, CACHE)
rdata = mujoco.MjData(rmodel)
renderer = mujoco.Renderer(rmodel, height=360, width=480)

def frame(d, camera="track"):
    rdata.qpos[:] = d.qpos
    mujoco.mj_forward(rmodel, rdata)
    renderer.update_scene(rdata, camera=camera)
    return renderer.render().copy()

def show(frames, fps=25):
    media.show_video(np.asarray(frames), fps=fps, codec="h264")

print("mujoco", mujoco.__version__, "| model:", paths.SCENE_XML.relative_to(REPO_ROOT),
      f"| siatki do rysowania: {int(rmodel.mesh_facenum.sum()):,} trójkątów")

## Krok 1 — Wojtek stoi w MuJoCo

Robot startuje z zapisanej pozy `home` i trzyma jej kąty w serwach. Nic więcej: tak wygląda „zerowa akcja” polityki.

In [ ]:
model = mujoco.MjModel.from_xml_path(str(paths.SCENE_XML))
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, model.key("home").id)
data.ctrl[:] = model.key("home").ctrl

frames = []
for k in range(int(4.0 / model.opt.timestep)):       # 4 s
    mujoco.mj_step(model, data)
    if k % 10 == 0:                                   # 25 klatek/s
        frames.append(frame(data))

print(f"serwa: {model.nu}, masa {sum(model.body_mass):.1f} kg, wysokość bazy {data.qpos[2]:.3f} m")
show(frames)

## Krok 2 — Polityka na początku treningu

Polityka to sieć neuronowa: na wejściu obserwacja, na wyjściu 12 przesunięć kątów względem pozy `home`. Obserwacja jest tym, co robot naprawdę mierzy: kąty i prędkości 12 przegubów, poprzednia akcja i **komenda** `[vx, vy, wz, wysokość]`, czyli wektor, w którym ma iść. Aktor ma warstwy 512-256-128, a w treningu do jego wyjścia dokłada się losowy szum, żeby próbował różnych ruchów.

Tak wygląda start PPO: te same wejścia, ta sama architektura, losowe wagi i szum. Komenda: 0,5 m/s do przodu. Sieć jeszcze nie wie, co komenda znaczy.

In [ ]:
KOMENDA = np.array([0.5, 0.0, 0.0, 0.125], np.float32)   # vx, vy, wz, wysokość stania
SEED = 0

model = mujoco.MjModel.from_xml_path(str(paths.SCENE_XML))
model.actuator_gainprm[:, 0], model.actuator_biasprm[:, 1], model.actuator_biasprm[:, 2] = 40.0, -40.0, -1.6   # serwo jak w treningu
model.actuator_forcerange[:] = [-9.0, 9.0]
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, model.key("home").id)
HOME = model.key("home").ctrl.copy()
qadr = np.array([model.jnt_qposadr[j] for j in model.actuator_trnid[:, 0]])
vadr = np.array([model.jnt_dofadr[j] for j in model.actuator_trnid[:, 0]])
SCALE = np.tile([0.25, 0.5, 0.5], 4)                      # zakres akcji na przegub, rad
LOW, HIGH = model.actuator_ctrlrange.T.copy()
LOW[0::3], HIGH[0::3] = -0.44, 0.44                        # limit odwodzenia i kolana jak w treningu
HIGH[2::3] = np.minimum(HIGH[2::3], 3.15)

# Aktor jak w PPO na starcie: losowe wagi, wyjście = środek i rozrzut rozkładu akcji.
rng = np.random.default_rng(SEED)
sizes = [12 + 12 + 12 + 4, 512, 256, 128, 2 * 12]
W = [rng.uniform(-1, 1, (a, b)) * np.sqrt(3.0 / a) for a, b in zip(sizes[:-1], sizes[1:])]
def aktor(obs):
    x = obs
    for w in W[:-1]:
        x = x @ w; x = x / (1 + np.exp(-x))               # SiLU
    out = x @ W[-1]
    std = np.log1p(np.exp(out[12:])) + 1e-3               # softplus, jak w Brax
    return np.tanh(out[:12] + std * rng.standard_normal(12))

frames, last_act = [], np.zeros(12, np.float32)
x0 = data.qpos[0]
for i in range(200):                                       # 4 s przy 50 Hz
    obs = np.concatenate([data.qpos[qadr] - HOME, data.qvel[vadr], last_act, KOMENDA])
    last_act = aktor(obs).astype(np.float32)
    data.ctrl[:] = np.clip(HOME + last_act * SCALE, LOW, HIGH)
    for _ in range(5):
        mujoco.mj_step(model, data)
    if i % 2 == 0:
        frames.append(frame(data))
print(f"po 4 s: przebyte {data.qpos[0] - x0:+.2f} m w kierunku komendy (cel: +2.0 m), wysokość bazy {data.qpos[2]:.3f} m")
show(frames)

## Krok 3 — Nagroda

Trening nie mówi sieci, *jak* chodzić. Mówi tylko, ile punktów dostaje za każdy krok 20 ms, a PPO zmienia wagi tak, żeby suma punktów w epizodzie rosła. Nagroda to suma składników: `dt · Σ waga · składnik`. Co można do niej włożyć:

- **Zadanie**: `tracking_lin_vel`, `tracking_ang_vel` — jak blisko prędkość robota jest komendy (kernel `exp(-błąd²/σ)`: 1 przy trafieniu, 0 daleko). To jedyne miejsce, gdzie komenda w ogóle ma znaczenie.
- **Postawa**: `orientation` (kara za przechył; aktor nie ma IMU, więc to jego jedyny „zmysł” pionu), `pose` (odchylenie od pozy stania), `stand_still` i `stand_feet_down` (ruch i uniesione stopy przy komendzie „stój”).
- **Chód**: `feet_air_time` i `high_step` (nagroda za odrywanie i unoszenie stóp), `feet_slip` (kara za poślizg stopy na ziemi).
- **Wysiłek i gładkość**: `torques`, `torque_rate`, `torque_limit` (moment, jego skoki, dobijanie do limitu), `action_rate` (skoki celów między krokami).
- **Koniec**: `termination` — kara za upadek, który kończy epizod.

Jak to wpływa na trening, w skrócie:

- Wagi to kompromis. Sam tracking daje robota, który drży i grzeje silniki; sama gładkość daje robota, który stoi. Każdy składnik ma cenę w innych.
- Sieć znajdzie luki. Za dużą karę za lądowanie omija, sunąc stopami zamiast stawiać kroki; kary za każdy krok potrafi „uniknąć” przewracając się wcześnie, bo krótszy epizod to mniej kar. Dlatego obok nagrody patrzy się na długość epizodu.
- Czego nie ma w nagrodzie, tego nie będzie w chodzie: obroty w miejscu czy chód do tyłu pojawiają się dopiero, gdy komendy je zadają, a nagroda za nie płaci.

Poniżej wagi z przepisu `locomotion_stiff_v1`, na którym za chwilę ruszy trening. Zero oznacza składnik wyłączony.

In [ ]:
from hydra import compose, initialize_config_dir

PRESET = "locomotion_stiff_v1"      # przepis opublikowanej polityki: bez IMU, serwo kp40/kd1.6/9 N·m
RUN_NAME = "guide_stiff_v1"
TRAIN_STEPS = 10_000_000            # próbka; pełny przepis to 2 000 000 000
NUM_ENVS = 2048                     # robotów liczonych naraz na GPU
OVERRIDES = [f"+experiment={PRESET}", f"run_name={RUN_NAME}", "seed=0",
             f"++ppo.num_timesteps={TRAIN_STEPS}", f"++ppo.num_envs={NUM_ENVS}", "wandb.enable=false"]
with initialize_config_dir(config_dir=str(TRAINING / "wojtek_rl" / "conf"), version_base=None):
    hcfg = compose(config_name="config", overrides=OVERRIDES)

GRUPY = {"zadanie": ["tracking_lin_vel", "tracking_ang_vel", "height_tracking"],
         "postawa": ["orientation", "pose", "stand_still", "stand_feet_down", "lin_vel_z", "ang_vel_xy"],
         "chód": ["feet_air_time", "high_step", "feet_slip", "feet_apex", "feet_landing"],
         "wysiłek": ["torques", "torque_rate", "torque_limit", "action_rate", "action_accel", "energy"],
         "koniec": ["termination"]}
scales = hcfg.task.env.reward.scales
for grupa, nazwy in GRUPY.items():
    print(f"{grupa:9s}", "  ".join(f"{n}={scales[n]:g}" for n in nazwy if n in scales))

## Krok 4 — Trening (GPU)

Ta sama sieć co w kroku 2, ale teraz 2048 robotów naraz na GPU, każdy 20 s epizodu, a po każdej porcji kroków PPO poprawia wagi w stronę większej nagrody. Co jakiś czas trener ocenia politykę: `reward` to suma nagrody z epizodu, `ep_len` to jego długość w krokach (1000 = nie upadł). 10 mln kroków to kilka-kilkanaście minut na T4; to za mało na dobry chód, wystarczająco, żeby zobaczyć różnicę względem kroku 2.

Jeśli MJWarp nie obsłuży karty, dopisz do `OVERRIDES`: `+task.env.sim.backend=jax`.

In [ ]:
import subprocess, re, jax
RUN_DIR = TRAINING / "runs" / RUN_NAME
has_gpu = any(d.platform == "gpu" for d in jax.devices())
if (RUN_DIR / "run.json").exists():
    print("trening już jest:", RUN_DIR)
elif not has_gpu:
    print("brak GPU: Runtime → Change runtime type → GPU")
else:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    with open(RUN_DIR / "train.log", "w") as log:
        proc = subprocess.Popen([sys.executable, "-m", "wojtek_rl.train", *OVERRIDES], cwd=TRAINING,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            if line.startswith(("steps", "done")) or "Error" in line:
                print(line, end="")
        proc.wait()
    print("koniec, kod", proc.returncode)

## Krok 5 — Wytrenowana polityka w MuJoCo

Eksport zamienia checkpoint na `policy.npz` (wagi) i `policy_meta.json` (kontrakt: układ obserwacji, skala akcji, limity, serwo). Dokładnie ten plik dostaje robot. Poniżej ta sama pętla co w kroku 2, ta sama komenda, tylko wagi po treningu.

In [ ]:
EXPORT_DIR = RUN_DIR / "deploy"
if not (RUN_DIR / "run.json").exists():
    raise SystemExit("najpierw krok 4 (trening)")
if not (EXPORT_DIR / "policy_meta.json").exists():
    res = subprocess.run([sys.executable, "-m", "wojtek_rl.export_policy", "--run", f"runs/{RUN_NAME}"],
                         cwd=TRAINING, env=dict(os.environ, JAX_PLATFORMS="cpu"), capture_output=True, text=True)
    print((res.stdout or res.stderr)[-600:])

from wojtek_rl.np_policy import load_policy_runtime
polityka = load_policy_runtime(EXPORT_DIR)
pd = polityka.meta["pd"]
model = mujoco.MjModel.from_xml_path(str(paths.SCENE_XML))
model.actuator_gainprm[:, 0], model.actuator_biasprm[:, 1], model.actuator_biasprm[:, 2] = pd["kp"], -pd["kp"], -pd["kd"]
model.actuator_forcerange[:] = [-pd["max_torque"], pd["max_torque"]]
data = mujoco.MjData(model)
mujoco.mj_resetDataKeyframe(model, data, model.key("home").id)
polityka.reset()

frames, x0, zero3 = [], data.qpos[0], np.zeros(3, np.float32)
for i in range(200):                                       # 4 s przy 50 Hz
    data.ctrl[:] = polityka.step(zero3, zero3, data.qpos[qadr], data.qvel[vadr], KOMENDA)   # IMU nieużywane
    for _ in range(5):
        mujoco.mj_step(model, data)
    if i % 2 == 0:
        frames.append(frame(data))
print(f"po 4 s: przebyte {data.qpos[0] - x0:+.2f} m w kierunku komendy (cel: +2.0 m), wysokość bazy {data.qpos[2]:.3f} m")
show(frames)